# Phase 1 — Data Collection
## Financial News Sentiment Intelligence System

**Goal:** Build a robust data collection pipeline that fetches
real financial news from Yahoo Finance for 5 companies and saves
it to a structured CSV for downstream processing.

**Output:** `data/raw/all_news.csv`

**Why this matters:** Every ML model is only as good as its data.
Before we can score sentiment or predict prices, we need clean,
validated, well-structured raw data. This notebook builds that foundation.

In [5]:
# ── environment check ─────────────────────────────────────
# why: confirms everything is installed before running
# the full pipeline. Saves you from a crash halfway through.
# This is called a "preflight check" — pilots do it,

import sys
import os

# add project root to path so notebook can find src/ and config.py
# notebooks/ is a subfolder, so we go one level up with ".."
sys.path.insert(0, os.path.abspath(".."))

# import our modules
from config import COMPANIES, RAW_DATA_PATH
from src.utils import get_logger
from src.data_collector import collect_all_companies

# check Python version
print(f"Python version : {sys.version.split()[0]}")

# check all key libraries
import yfinance, pandas, numpy
print(f"yfinance       : {yfinance.__version__}")
print(f"pandas         : {pandas.__version__}")
print(f"numpy          : {numpy.__version__}")

# confirm config loaded
print(f"\nCompanies      : {COMPANIES}")
print(f"Save path      : {RAW_DATA_PATH}")

print("\n✅ Preflight check passed. Ready to collect data.")

Python version : 3.11.7
yfinance       : 1.3.0
pandas         : 3.0.3
numpy          : 2.4.4

Companies      : ['AAPL', 'GOOGL', 'META', 'AMZN', 'MSFT']
Save path      : data\raw\all_news.csv

✅ Preflight check passed. Ready to collect data.


## Step 1 — Run the collection pipeline

Calling `collect_all_companies()` from `src/data_collector.py`.

This single function call runs the entire pipeline:
fetch → parse → validate → save.

Watch the logs below — they tell you exactly what is happening
at each step. Notice the rejection rate — this is your first
data quality metric.

In [2]:
# ── run the full collection pipeline ──────────────────────
# one function call — that is the power of building src/ modules

df = collect_all_companies()

print(f"\n{'='*50}")
print(f"Collection complete.")
print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")

2026-05-14 04:44:33 | src.data_collector   | INFO     | =======================================================
2026-05-14 04:44:33 | src.data_collector   | INFO     | Starting data collection pipeline
2026-05-14 04:44:33 | src.data_collector   | INFO     | Companies: ['AAPL', 'GOOGL', 'META', 'AMZN', 'MSFT']
2026-05-14 04:44:33 | src.data_collector   | INFO     | =======================================================
2026-05-14 04:44:34 | src.data_collector   | INFO     | Fetched 10 raw articles for AAPL
2026-05-14 04:44:34 | src.data_collector   | INFO     | Fetched 10 raw articles for GOOGL
2026-05-14 04:44:34 | src.data_collector   | INFO     | Fetched 10 raw articles for META
2026-05-14 04:44:34 | src.data_collector   | INFO     | Fetched 10 raw articles for AMZN
2026-05-14 04:44:34 | src.data_collector   | INFO     | Fetched 10 raw articles for MSFT
2026-05-14 04:44:35 | src.data_collector   | INFO     | 
── Collection Summary ──────────────────
--- Logging error ---
Traceback (


Collection complete.
Dataset shape: 50 rows × 6 columns


## Step 2 — Inspect the raw data

Before saving any data to processed/, we always inspect it.
The questions we ask:
- Does the shape match expectations?
- Are there missing values?
- Are the dates in the right format?
- Do the headlines look like real financial news?

This inspection is what you describe in interviews when asked
"how do you validate incoming data?"

In [3]:
import pandas as pd

# reload from CSV — confirms the save worked correctly
df = pd.read_csv(f"../{RAW_DATA_PATH}")

print("─── SHAPE ────────────────────────────────")
print(f"  {df.shape[0]} rows × {df.shape[1]} columns\n")

print("─── COLUMNS ──────────────────────────────")
for col in df.columns:
    dtype = df[col].dtype
    nulls = df[col].isnull().sum()
    print(f"  {col:<12} dtype={str(dtype):<10} nulls={nulls}")

print("\n─── ARTICLES PER COMPANY ─────────────────")
print(df.groupby("ticker").size().to_string())

print("\n─── DATE RANGE ───────────────────────────")
print(f"  Earliest : {df['date'].min()}")
print(f"  Latest   : {df['date'].max()}")

print("\n─── SAMPLE HEADLINES ─────────────────────")
for _, row in df.sample(5, random_state=42).iterrows():
    print(f"  [{row['ticker']}] {row['title'][:70]}")

print("\n─── FULL DATAFRAME PREVIEW ───────────────")
df.head(10)

─── SHAPE ────────────────────────────────
  50 rows × 6 columns

─── COLUMNS ──────────────────────────────
  ticker       dtype=str        nulls=0
  title        dtype=str        nulls=0
  summary      dtype=str        nulls=0
  date         dtype=str        nulls=0
  url          dtype=str        nulls=0
  date_clean   dtype=str        nulls=0

─── ARTICLES PER COMPANY ─────────────────
ticker
AAPL     10
AMZN     10
GOOGL    10
META     10
MSFT     10

─── DATE RANGE ───────────────────────────
  Earliest : 11 May 2026, 02:03 PM UTC
  Latest   : 11 May 2026, 02:03 PM UTC

─── SAMPLE HEADLINES ─────────────────────
  [GOOGL] Top Midday Stories: Moderna Working on Hantavirus Vaccine, Shares Rise
  [AMZN] AI Wins Have Alphabet Poised to Become World’s Biggest Company
  [AMZN] The $1 trillion club's new members are powering the AI boom: Chart of 
  [MSFT] Microsoft CEO Takes Stand in Third Week of Elon Musk Megatrial Against
  [GOOGL] Alphabet Taps Yen Bond Market To Support Expanding 

,ticker,title,summary,date,url,date_clean
0,AAPL,"Memory chip stocks hit record highs, pharma re...",Market Catalysts Host Julie Hyman and Yahoo Fi...,"11 May 2026, 02:03 PM UTC",https://finance.yahoo.com/video/memory-chip-st...,2026-05-11
1,AAPL,The $1 trillion club's new members are powerin...,Market royalty is getting a hardware makeover.,"11 May 2026, 02:03 PM UTC",https://finance.yahoo.com/markets/article/the-...,2026-05-11
2,AAPL,Apple delivers surprise win in Big Tech’s AI s...,Apple (AAPL) is spending much of the artificia...,"11 May 2026, 02:03 PM UTC",https://www.thestreet.com/investing/apple-deli...,2026-05-11
3,AAPL,Apple-Intel preliminary chip deal seen as posi...,"Apple Inc (NASDAQ:AAPL, XETRA:APC) and Intel C...","11 May 2026, 02:03 PM UTC",https://www.proactiveinvestors.com/companies/n...,2026-05-11
4,AAPL,Asset Manager Sells 1.2 Million Bond ETF Share...,This ETF tracks a portfolio of U.S. Treasury b...,"11 May 2026, 02:03 PM UTC",https://www.fool.com/coverage/filings/2026/05/...,2026-05-11
5,AAPL,Apple Gets a Warning From Nintendo Memory-Chip...,Nintendo stock dropped after it warned it woul...,"11 May 2026, 02:03 PM UTC",https://www.barrons.com/articles/apple-stock-m...,2026-05-11
6,AAPL,"Elon Musk, Apple's Cook and Boeing CEO going t...","Elon Musk, Apple's Tim Cook, GE Aerospace's La...","11 May 2026, 02:03 PM UTC",https://finance.yahoo.com/sectors/technology/a...,2026-05-11
7,AAPL,The Hidden Drag of SPY’s Outdated UIT Structur...,SPDR S&P 500 ETF Trust (NYSEARCA:SPY) and Vang...,"11 May 2026, 02:03 PM UTC",https://247wallst.com/investing/2026/05/11/the...,2026-05-11
8,AAPL,Should You Buy Apple Stock Before June 8?,The company's annual Worldwide Developers Conf...,"11 May 2026, 02:03 PM UTC",https://www.fool.com/investing/2026/05/11/shou...,2026-05-11
9,AAPL,Intel Stock Rises on Report of Second Huge Chi...,Intel stock was gaining on a report it could b...,"11 May 2026, 02:03 PM UTC",https://www.barrons.com/articles/intel-stock-p...,2026-05-11


## Step 3 — Data quality assessment

A professional data quality check asks three questions:

1. **Completeness** — are all fields populated?
2. **Consistency** — are dates in the same format? Tickers uppercase?
3. **Validity** — do headlines look like real financial news?

Document your findings below.

In [4]:
print("─── COMPLETENESS ─────────────────────────")
missing = df.isnull().sum()
total   = len(df)
for col, count in missing.items():
    pct = count / total * 100
    status = "✅" if count == 0 else "⚠️ "
    print(f"  {status} {col:<12}: {count} missing ({pct:.1f}%)")

print("\n─── CONSISTENCY ──────────────────────────")
# are all tickers uppercase?
tickers_clean = df["ticker"].str.isupper().all()
print(f"  {'✅' if tickers_clean else '❌'} All tickers uppercase: {tickers_clean}")

# are all dates in YYYY-MM-DD format?
date_format_ok = df["date"].str.match(r"\d{4}-\d{2}-\d{2}").all()
print(f"  {'✅' if date_format_ok else '❌'} All dates YYYY-MM-DD: {date_format_ok}")

# any duplicate headlines?
dupes = df.duplicated(subset=["title"]).sum()
print(f"  {'✅' if dupes == 0 else '⚠️ '} Duplicate headlines: {dupes}")

print("\n─── VALIDITY ─────────────────────────────")
# headline word count distribution
df["title_words"] = df["title"].str.split().str.len()
print(f"  Headline word count:")
print(f"    min    : {df['title_words'].min()}")
print(f"    mean   : {df['title_words'].mean():.1f}")
print(f"    max    : {df['title_words'].max()}")
short = (df["title_words"] < 5).sum()
print(f"  Headlines under 5 words: {short}")

─── COMPLETENESS ─────────────────────────
  ✅ ticker      : 0 missing (0.0%)
  ✅ title       : 0 missing (0.0%)
  ✅ summary     : 0 missing (0.0%)
  ✅ date        : 0 missing (0.0%)
  ✅ url         : 0 missing (0.0%)
  ✅ date_clean  : 0 missing (0.0%)

─── CONSISTENCY ──────────────────────────
  ✅ All tickers uppercase: True
  ❌ All dates YYYY-MM-DD: False
  ⚠️  Duplicate headlines: 16

─── VALIDITY ─────────────────────────────
  Headline word count:
    min    : 7
    mean   : 11.8
    max    : 22
  Headlines under 5 words: 0


## Step 4 — Save and commit

Raw data saved to `data/raw/all_news.csv`.

**Key decisions documented:**
- Raw data is saved exactly as collected — never modified
- Validation happened before saving (rejected articles logged)
- One CSV per collection run — we will append in production

**Next phase:** EDA — we will explore patterns in this data,
visualise headline sentiment themes, and prepare it for
FinBERT scoring.